# Guideline Full-Text Sex-Based Characteristics Analysis Pipeline
================================================================

This notebook analyzes the full text of clinical guidelines (PDFs) to identify
sex-based characteristics and considerations, complementing the citation-level
analysis already performed on guideline references.

Pipeline Overview:
------------------
1. Load guideline metadata from CSV (PMID → PDF filename mapping)
2. Extract text from PDFs with page tracking
3. Search for sex-based characteristics using same patterns as citation analysis
4. Capture snippets with page numbers and context
5. Generate analysis results with evidence snippets
6. Output: CSV with guideline-level findings + detailed snippet evidence

Input Files:
-----------
- data/final_guidelines.csv (contains guideline_pmid and pdf_filename columns)
- data/guidelines full text/*.pdf (PDF files of guidelines)

Output Files:
------------
- output/guideline_fulltext_sex_analysis.csv (main results)
- output/guideline_fulltext_snippets.csv (detailed snippet evidence)
- output/guideline_fulltext_processing_log.txt (processing log)
"""















In [1]:
# ============================================================================
# CONFIGURATION SECTION
# ============================================================================

import os
import sys
import re
from datetime import datetime
from typing import List, Dict, Optional, Any, Pattern, Tuple
import logging

# ----------------------------------------------------------------------------
# Data Processing & Analysis
# ----------------------------------------------------------------------------
import pandas as pd
import numpy as np

# ----------------------------------------------------------------------------
# Progress Bars & Visualization
# ----------------------------------------------------------------------------
from tqdm.notebook import tqdm

# ----------------------------------------------------------------------------
# Excel File Handling
# ----------------------------------------------------------------------------
import xlsxwriter
import openpyxl

# ============================================================================
# SETUP LOGGING
# ============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("="*80)
print("INITIALIZING CONFIGURATION")
print("="*80)

# ============================================================================
# PROJECT METADATA
# ============================================================================
PROJECT_NAME = "Guidelines Supporting Sex Based Characteristics Full Text Analysis"
PROJECT_SHORT_NAME = "sex_characteristics"
RUN_DATE = datetime.now().strftime("%Y%m%d")

# ============================================================================
# INPUT FILES & FOLDERS
# ============================================================================
INPUT_DATA_FOLDER = 'data'
INPUT_CSV_FILENAME = 'all_final_guidelines.csv'
INPUT_PDF_FOLDER = 'guidelines full text'

# Full paths
INPUT_CSV_PATH = os.path.join(INPUT_DATA_FOLDER, INPUT_CSV_FILENAME)
PDF_DIRECTORY = os.path.join(INPUT_DATA_FOLDER, INPUT_PDF_FOLDER)
PDF_FOLDER = PDF_DIRECTORY  # Alias

# ============================================================================
# COLUMN NAMES - Input CSV
# ============================================================================
COL_GUIDELINE_ID = 'PMID'
COL_PDF_FILENAME = 'PDF File Name'
COL_GUIDELINE_TITLE = 'Title'
COL_JOURNAL = 'Journal/Book'
COL_YEAR = 'Publication Year'

REQUIRED_INPUT_COLS = [COL_GUIDELINE_ID, COL_PDF_FILENAME]

# ============================================================================
# OUTPUT FOLDERS & FILES
# ============================================================================
OUTPUT_FOLDER = os.path.join('output', 'all_final_guidelines', 'full_text_analysis')
CHECKPOINT_FOLDER = os.path.join(OUTPUT_FOLDER, 'checkpoints')

# Output file paths

# Output file paths
ANALYSIS_CSV_FILENAME = 'guideline_fulltext_sex_analysis.csv'  # ← Added "guideline_"
ANALYSIS_CSV_PATH = os.path.join(OUTPUT_FOLDER, ANALYSIS_CSV_FILENAME)

SNIPPETS_CSV_FILENAME = 'guideline_fulltext_snippets.csv'  # ← Added "guideline_"
SNIPPETS_CSV_PATH = os.path.join(OUTPUT_FOLDER, SNIPPETS_CSV_FILENAME)

SUMMARY_TXT_FILENAME = 'guideline_fulltext_summary.txt'
SUMMARY_TXT_PATH = os.path.join(OUTPUT_FOLDER, SUMMARY_TXT_FILENAME)

EXCEL_FILENAME = 'guideline_fulltext_analysis_COMPLETE.xlsx'
EXCEL_PATH = os.path.join(OUTPUT_FOLDER, EXCEL_FILENAME)

LOG_FILENAME = f'{PROJECT_SHORT_NAME}_processing_log.txt'
LOG_PATH = os.path.join(OUTPUT_FOLDER, LOG_FILENAME)

# Backward-compatible aliases
ANALYSIS_FILE = ANALYSIS_CSV_PATH
SNIPPETS_FILE = SNIPPETS_CSV_PATH
SUMMARY_FILE = SUMMARY_TXT_PATH
EXCEL_FILE = EXCEL_PATH

# ============================================================================
# ANALYSIS PARAMETERS
# ============================================================================
MAX_SNIPPETS_PER_MATCH = 50
CONTEXT_CHARS = 150
ENABLE_SECTION_DETECTION = True
MAX_WORKERS = 4
BATCH_SIZE = 100
ENABLE_CHECKPOINTS = True
CHECKPOINT_FREQUENCY = 10

# ============================================================================
# OUTPUT COLUMN NAMES
# ============================================================================
OUTPUT_COLS_ANALYSIS = {
    'guideline_id': COL_GUIDELINE_ID,
    'pdf_filename': COL_PDF_FILENAME,
    'total_pages': 'total_pages',
    'sex_based_found': 'sex_based_characteristics_found',
    'pregnancy_found': 'pregnancy_related_found',
    'sex_specific_found': 'sex_specific_conditions_found',
    'match_count': 'total_matches',
    'match_pages': 'pages_with_matches',
    'processing_status': 'status',
    'error_message': 'error'
}

OUTPUT_COLS_SNIPPETS = {
    'guideline_id': COL_GUIDELINE_ID,
    'category': 'pattern_category',
    'page': 'page_num',
    'section': 'section',
    'matched_text': 'matched_text',
    'snippet': 'snippet_text'
}

# ============================================================================
# SECTION PATTERNS (for detecting document sections)
# ============================================================================
# These will be populated after PATTERN_CATEGORIES is defined
# Placeholder for now - will be updated below
SECTION_PATTERNS = {
    'methods': re.compile(r'\bmethods?\b', re.IGNORECASE),
    'results': re.compile(r'\bresults?\b', re.IGNORECASE),
    'discussion': re.compile(r'\bdiscussion\b', re.IGNORECASE),
    'conclusions': re.compile(r'\bconclusions?\b', re.IGNORECASE),
    'recommendations': re.compile(r'\brecommendations?\b', re.IGNORECASE),
}

PATTERN_CATEGORIES = {}
HIGH_CONFIDENCE_PATTERNS = []

# ============================================================================
# COMPREHENSIVE CONFIG DICTIONARY
# ============================================================================


CONFIG = {
    # ========================================================================
    # ANALYSIS PARAMETERS
    # ========================================================================
    "max_snippets_per_pattern": MAX_SNIPPETS_PER_MATCH,
    "snippet_context_chars": CONTEXT_CHARS,
    "section_detection": ENABLE_SECTION_DETECTION,
    "max_workers": MAX_WORKERS,
    "batch_size": BATCH_SIZE,
    "enable_checkpoints": ENABLE_CHECKPOINTS,
    "checkpoint_frequency": CHECKPOINT_FREQUENCY,
    
    # ========================================================================
    # PATHS
    # ========================================================================
    "input_csv_path": INPUT_CSV_PATH,
    "pdf_directory": PDF_DIRECTORY,
    "pdf_folder": PDF_FOLDER,  # Alias
    "output_folder": OUTPUT_FOLDER,
    "checkpoint_folder": CHECKPOINT_FOLDER,
    "analysis_csv_path": ANALYSIS_CSV_PATH,
    "snippets_csv_path": SNIPPETS_CSV_PATH,
    "summary_txt_path": SUMMARY_TXT_PATH,
    "excel_path": EXCEL_PATH,
    "log_path": LOG_PATH,
    
    # ========================================================================
    # COLUMN NAMES
    # ========================================================================
    "col_guideline_id": COL_GUIDELINE_ID,
    "col_pdf_filename": COL_PDF_FILENAME,
    "col_guideline_title": COL_GUIDELINE_TITLE,
    "col_journal": COL_JOURNAL,
    "col_year": COL_YEAR,
    "required_input_cols": REQUIRED_INPUT_COLS,
    
    # ========================================================================
    # OUTPUT COLUMN MAPPINGS
    # ========================================================================
    "output_cols_analysis": OUTPUT_COLS_ANALYSIS,
    "output_cols_snippets": OUTPUT_COLS_SNIPPETS,
    
    # ========================================================================
    # PATTERN DEFINITIONS (will be updated after patterns are defined)
    # ========================================================================
    "section_patterns": SECTION_PATTERNS,
    "pattern_categories": PATTERN_CATEGORIES,  # Will be updated
    "high_confidence_patterns": HIGH_CONFIDENCE_PATTERNS,  # Will be updated
    
    # ========================================================================
    # PROJECT METADATA
    # ========================================================================
    "project_name": PROJECT_NAME,
    "project_short_name": PROJECT_SHORT_NAME,
    "run_date": RUN_DATE,
}


# ============================================================================
# PDF Library Detection
# ============================================================================

# Try to import PDF libraries
PDF_LIBRARY = None

try:
    import fitz  # PyMuPDF
    PDF_LIBRARY = 'pymupdf'
    print("✓ Using PyMuPDF for PDF extraction")
except ImportError:
    try:
        import pdfplumber
        PDF_LIBRARY = 'pdfplumber'
        print("✓ Using pdfplumber for PDF extraction")
    except ImportError:
        PDF_LIBRARY = None
        print("❌ No PDF library available!")
        print("Please install one: pip install PyMuPDF OR pip install pdfplumber")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def get_config(key: str, default: Any = None) -> Any:
    """
    Safely get a configuration value.
    
    Args:
        key: Configuration key
        default: Default value if key not found
    
    Returns:
        Configuration value or default
    
    Example:
        max_snippets = get_config('max_snippets_per_pattern', 5)
    """
    return CONFIG.get(key, default)


def update_config(key: str, value: Any) -> None:
    """
    Update a configuration value.
    
    Args:
        key: Configuration key
        value: New value
    
    Example:
        update_config('max_snippets_per_pattern', 10)
    """
    CONFIG[key] = value
    logger.info(f"Updated config: {key} = {value}")


def update_config_patterns(pattern_categories: Dict[str, List[Pattern]], 
                          high_confidence_patterns: List[str]) -> None:
    """
    Update CONFIG with pattern definitions after they're created.
    
    IMPORTANT: Call this after defining your patterns!
    
    Args:
        pattern_categories: Dictionary of pattern categories
        high_confidence_patterns: List of high-confidence pattern names
    
    Example:
        update_config_patterns(PATTERN_CATEGORIES, HIGH_CONFIDENCE_PATTERNS)
    """
    global PATTERN_CATEGORIES, HIGH_CONFIDENCE_PATTERNS  
    
    PATTERN_CATEGORIES = pattern_categories
    HIGH_CONFIDENCE_PATTERNS = high_confidence_patterns
    
    CONFIG['pattern_categories'] = pattern_categories
    CONFIG['high_confidence_patterns'] = high_confidence_patterns
    
    logger.info(f"Updated CONFIG with {len(pattern_categories)} pattern categories")
    logger.info(f"High confidence patterns: {high_confidence_patterns}")


def get_column_name(standard_name: str) -> str:
    """
    Get the actual column name from standard naming.
    
    Args:
        standard_name: Standard name (e.g., 'guideline_id')
    
    Returns:
        Actual column name (e.g., 'PMID')
    
    Example:
        id_col = get_column_name('guideline_id')  # Returns 'PMID'
    """
    column_map = {
        'guideline_id': COL_GUIDELINE_ID,
        'pdf_filename': COL_PDF_FILENAME,
        'title': COL_GUIDELINE_TITLE,
        'journal': COL_JOURNAL,
        'year': COL_YEAR
    }
    return column_map.get(standard_name, standard_name)


def validate_config() -> bool:
    """
    Validate configuration settings.
    
    Returns:
        True if valid, raises ValueError if invalid
    """
    # Check required parameters
    required_keys = [
        'max_snippets_per_pattern',
        'snippet_context_chars',
        'col_guideline_id',
        'col_pdf_filename',
        'input_csv_path',
        'output_folder'
    ]
    
    missing_keys = [k for k in required_keys if k not in CONFIG]
    if missing_keys:
        raise ValueError(f"CONFIG missing required keys: {missing_keys}")
    
    # Validate numeric parameters
    if CONFIG['max_snippets_per_pattern'] < 1:
        raise ValueError("max_snippets_per_pattern must be >= 1")
    if CONFIG['snippet_context_chars'] < 0:
        raise ValueError("snippet_context_chars must be >= 0")
    
    logger.info("✓ Configuration validated successfully")
    return True


def setup_folders() -> bool:
    """
    Create all necessary output folders.
    
    Returns:
        True if successful
    """
    folders_to_create = [OUTPUT_FOLDER, CHECKPOINT_FOLDER]
    
    for folder in folders_to_create:
        if not os.path.exists(folder):
            os.makedirs(folder, exist_ok=True)
            print(f"✓ Created folder: {folder}")
        else:
            print(f"✓ Folder exists: {folder}")
    
    return True


def print_config_summary() -> None:
    """Print a summary of current configuration."""
    print("=" * 80)
    print("CONFIGURATION SUMMARY")
    print("=" * 80)
    print(f"\n📁 INPUT FILES:")
    print(f"   CSV:        {INPUT_CSV_PATH}")
    print(f"   PDFs:       {PDF_DIRECTORY}")
    print(f"\n🔑 KEY COLUMNS:")
    print(f"   ID Column:  {COL_GUIDELINE_ID}")
    print(f"   PDF Column: {COL_PDF_FILENAME}")
    print(f"\n📤 OUTPUT LOCATION:")
    print(f"   Folder:     {OUTPUT_FOLDER}")
    print(f"   Analysis:   {ANALYSIS_CSV_FILENAME}")
    print(f"   Snippets:   {SNIPPETS_CSV_FILENAME}")
    print(f"   Excel:      {EXCEL_FILENAME}")
    print(f"\n⚙️  PARAMETERS:")
    print(f"   Max snippets:        {MAX_SNIPPETS_PER_MATCH}")
    print(f"   Context chars:       {CONTEXT_CHARS}")
    print(f"   Section detection:   {ENABLE_SECTION_DETECTION}")
    print(f"   Checkpoint enabled:  {ENABLE_CHECKPOINTS}")
    print(f"   Batch size:          {BATCH_SIZE}")
    
    # Pattern info
    num_pattern_cats = len(CONFIG.get('pattern_categories', {}))
    num_high_conf = len(CONFIG.get('high_confidence_patterns', []))
    print(f"\n📊 PATTERNS:")
    print(f"   Pattern categories:  {num_pattern_cats}")
    print(f"   High confidence:     {num_high_conf}")
    
    print("=" * 80)


# ============================================================================
# INITIALIZATION
# ============================================================================

# Create folders
setup_folders()

# Validate configuration
validate_config()

# Print summary
print_config_summary()

print("\n✅ Configuration initialized successfully!")
print("✅ All folders created/verified!")
print("✅ CONFIG dictionary ready for use!")
print("\n⚠️  IMPORTANT: After defining your pattern categories in the next cell,")
print("   call: update_config_patterns(PATTERN_CATEGORIES, HIGH_CONFIDENCE_PATTERNS)")
print()

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def get_config_value(config: Dict[str, Any], key: str, default: Any = None) -> Any:
    """
    Safely get a configuration value.
    
    Args:
        config: Configuration dictionary
        key: Key to retrieve
        default: Default value if key not found
    
    Returns:
        Value from config or default
    """
    return config.get(key, default)

2026-02-05 19:49:15,529 - INFO - ✓ Configuration validated successfully


INITIALIZING CONFIGURATION
✓ Using PyMuPDF for PDF extraction
✓ Folder exists: output\all_final_guidelines\full_text_analysis
✓ Folder exists: output\all_final_guidelines\full_text_analysis\checkpoints
CONFIGURATION SUMMARY

📁 INPUT FILES:
   CSV:        data\all_final_guidelines.csv
   PDFs:       data\guidelines full text

🔑 KEY COLUMNS:
   ID Column:  PMID
   PDF Column: PDF File Name

📤 OUTPUT LOCATION:
   Folder:     output\all_final_guidelines\full_text_analysis
   Analysis:   guideline_fulltext_sex_analysis.csv
   Snippets:   guideline_fulltext_snippets.csv
   Excel:      guideline_fulltext_analysis_COMPLETE.xlsx

⚙️  PARAMETERS:
   Max snippets:        50
   Context chars:       150
   Section detection:   True
   Checkpoint enabled:  True
   Batch size:          100

📊 PATTERNS:
   Pattern categories:  0
   High confidence:     0

✅ Configuration initialized successfully!
✅ All folders created/verified!
✅ CONFIG dictionary ready for use!

⚠️  IMPORTANT: After defining your pat

In [2]:
# ============================================================================
# Regex Patterns (Same as Citation Analysis Pipeline)
# ============================================================================

# Sex differences and mentions
SEX_DIFF_PATTERNS = [
    re.compile(r'\bsex\b', re.IGNORECASE),
    re.compile(r'\bgender\b', re.IGNORECASE),
    re.compile(r'\bmale[s]?\b', re.IGNORECASE),
    re.compile(r'\bfemale[s]?\b', re.IGNORECASE),
    re.compile(r'\b(wo)?men\b', re.IGNORECASE),
    re.compile(r'\bsex[- ]specific\b', re.IGNORECASE),
    re.compile(r'\bgender[- ]specific\b', re.IGNORECASE),
    re.compile(r'\bsex[- ]based\b', re.IGNORECASE),
    re.compile(r'\bgender[- ]based\b', re.IGNORECASE),
    re.compile(r'\bsex\s+(difference|differences|disparity|disparities)\b', re.IGNORECASE),
    re.compile(r'\bgender\s+(difference|differences|disparity|disparities)\b', re.IGNORECASE),
    re.compile(r'\bby sex\b', re.IGNORECASE),
    re.compile(r'\baccording to sex\b', re.IGNORECASE),
    re.compile(r'\bbetween.*sexes\b', re.IGNORECASE),
    re.compile(r'\bsex[- ]disaggregated\b', re.IGNORECASE),
    re.compile(r'\bsex[- ]stratified\b', re.IGNORECASE),
    re.compile(r'\bgender[- ]stratified\b', re.IGNORECASE),
    re.compile(r'\bsex as.*variable\b', re.IGNORECASE),
    re.compile(r'\bgender as.*variable\b', re.IGNORECASE),
    re.compile(r'\bsex.*analysis\b', re.IGNORECASE),
    re.compile(r'\bgender.*analysis\b', re.IGNORECASE),
]

# Stratification patterns
STRAT_PATTERNS = [
    re.compile(r'\bstratif\w*\s+by\s+(sex|gender)\b', re.IGNORECASE),
    re.compile(r'\b(sex|gender)[- ]stratified\b', re.IGNORECASE),
    re.compile(r'\bstratification\s+by\s+(sex|gender)\b', re.IGNORECASE),
    re.compile(r'\banaly(?:s|z)ed\s+separately\s+(for|by)\s+(sex|gender|men and women)\b', re.IGNORECASE),
    re.compile(r'\bseparate\s+analyses?\s+(for|by)\s+(sex|gender|men and women)\b', re.IGNORECASE),
]

# Subgroup patterns
SUBGROUP_PATTERNS = [
    re.compile(r'\bsubgroup\s+analysis.*\b(sex|gender|men|women)\b', re.IGNORECASE),
    re.compile(r'\b(sex|gender|men|women)\b.*subgroup\s+analysis', re.IGNORECASE),
    re.compile(r'\bsubgroup.*by\s+(sex|gender)\b', re.IGNORECASE),
    re.compile(r'\b(sex|gender)\s+subgroup\b', re.IGNORECASE),
    re.compile(r'\binteraction.*\b(sex|gender)\b', re.IGNORECASE),
]

# Interaction patterns
INTERACTION_PATTERNS = [
    re.compile(r'\bsex.*interaction\b', re.IGNORECASE),
    re.compile(r'\bgender.*interaction\b', re.IGNORECASE),
    re.compile(r'\binteraction.*sex\b', re.IGNORECASE),
    re.compile(r'\binteraction.*gender\b', re.IGNORECASE),
    re.compile(r'\binteraction.*between.*sex\b', re.IGNORECASE),
]

# Pregnancy patterns
PREG_PATTERNS = [
    re.compile(r'\bpregnant\b', re.IGNORECASE),
    re.compile(r'\bpregnancy\b', re.IGNORECASE),
    re.compile(r'\bgestational\b', re.IGNORECASE),
    re.compile(r'\bantepartum\b', re.IGNORECASE),
    re.compile(r'\bpostpartum\b', re.IGNORECASE),
    re.compile(r'\bprenatal\b', re.IGNORECASE),
    re.compile(r'\bpostnatal\b', re.IGNORECASE),
]

# Menopause patterns
MENO_PATTERNS = [
    re.compile(r'\bmenopause\b', re.IGNORECASE),
    re.compile(r'\bmenopausal\b', re.IGNORECASE),
    re.compile(r'\bpostmenopausal\b', re.IGNORECASE),
    re.compile(r'\bperimenopausal\b', re.IGNORECASE),
]

# Contraception patterns
CONTRA_PATTERNS = [
    re.compile(r'\bcontraception\b', re.IGNORECASE),
    re.compile(r'\bcontraceptive\b', re.IGNORECASE),
    re.compile(r'\bbirth control\b', re.IGNORECASE),
]

# Hormonal patterns
HORM_PATTERNS = [
    re.compile(r'\bhormone replacement\b', re.IGNORECASE),
    re.compile(r'\bhormonal therapy\b', re.IGNORECASE),
    re.compile(r'\bestrogen\b', re.IGNORECASE),
    re.compile(r'\bprogesterone\b', re.IGNORECASE),
    re.compile(r'\btestosterone\b', re.IGNORECASE),
]

# Reproductive health patterns
REPRODUCTIVE_PATTERNS = [
    re.compile(r'\breproductive health\b', re.IGNORECASE),
    re.compile(r'\bfertility\b', re.IGNORECASE),
    re.compile(r'\binfertility\b', re.IGNORECASE),
    re.compile(r'\bovarian\b', re.IGNORECASE),
    re.compile(r'\buterine\b', re.IGNORECASE),
]

# Maternal/offspring patterns
MATERNAL_OFFSPRING_PATTERNS = [
    re.compile(r'\bmaternal\b', re.IGNORECASE),
    re.compile(r'\boffspring\b', re.IGNORECASE),
    re.compile(r'\bfetal\b', re.IGNORECASE),
    re.compile(r'\bneonatal\b', re.IGNORECASE),
]

# Lactation/breastfeeding patterns
LACTATION_BREAST_PATTERNS = [
    re.compile(r'\blactation\b', re.IGNORECASE),
    re.compile(r'\bbreastfeeding\b', re.IGNORECASE),
    re.compile(r'\bnursing\b', re.IGNORECASE),
]

# Women's health conditions
WOMENS_CONDITIONS_PATTERNS = [
    re.compile(r'\bendometriosis\b', re.IGNORECASE),
    re.compile(r'\bpolycystic ovary\b', re.IGNORECASE),
    re.compile(r'\bPCOS\b'),
    re.compile(r'\bcervical cancer\b', re.IGNORECASE),
    re.compile(r'\bovarian cancer\b', re.IGNORECASE),
    re.compile(r'\buterine cancer\b', re.IGNORECASE),
]

# Gender identity patterns
GENDER_IDENTITY_PATTERNS = [
    re.compile(r'\btransgender(ed)?\b', re.IGNORECASE),
    re.compile(r'\bgender dysphoria\b', re.IGNORECASE),
    re.compile(r'\bgender identit(y|ies)\b', re.IGNORECASE),
    re.compile(r'\bgender minorit(y|ies)\b', re.IGNORECASE),
    re.compile(r'\bgender[- ]diverse\b', re.IGNORECASE),
]

# Exclusion patterns
EXCLUSION_PREGNANT_PATTERNS = [
    re.compile(r'\bexclud(e|ed|ing).*pregnant', re.IGNORECASE),
    re.compile(r'\bpregnant.*excluded', re.IGNORECASE),
    re.compile(r'\bexclusion.*pregnancy', re.IGNORECASE),
]

EXCLUSION_CHILDBEARING_PATTERNS = [
    re.compile(r'\bexclud(e|ed|ing).*(childbearing|child-bearing)', re.IGNORECASE),
    re.compile(r'\b(childbearing|child-bearing).*excluded', re.IGNORECASE),
    re.compile(r'\bwomen of.*potential.*excluded', re.IGNORECASE),
]

# Pattern categories for analysis
PATTERN_CATEGORIES = {
    'sex_differences': SEX_DIFF_PATTERNS,
    'sex_stratification': STRAT_PATTERNS,
    'sex_subgroup': SUBGROUP_PATTERNS,
    'sex_interaction': INTERACTION_PATTERNS,
    'pregnancy_related': PREG_PATTERNS,
    'menopause_related': MENO_PATTERNS,
    'contraception_required': CONTRA_PATTERNS,
    'hormonal_related': HORM_PATTERNS,
    'reproductive_health': REPRODUCTIVE_PATTERNS,
    'maternal_offspring': MATERNAL_OFFSPRING_PATTERNS,
    'lactation_breast': LACTATION_BREAST_PATTERNS,
    'womens_conditions': WOMENS_CONDITIONS_PATTERNS,
    'gender_identity': GENDER_IDENTITY_PATTERNS,
    'excludes_pregnant_women': EXCLUSION_PREGNANT_PATTERNS,
    'excludes_childbearing_potential': EXCLUSION_CHILDBEARING_PATTERNS,
}

# High-confidence pattern categories for emphasis
# These indicate substantive sex considerations, not just background mentions
HIGH_CONFIDENCE_PATTERNS = [
    'sex_stratification',
    'sex_subgroup', 
    'sex_interaction',
    'excludes_pregnant_women',  # For transparency about exclusions
]

# Update CONFIG with pattern definitions
update_config_patterns(PATTERN_CATEGORIES, HIGH_CONFIDENCE_PATTERNS)
print("✓ Patterns loaded into CONFIG")

2026-02-05 19:49:15,554 - INFO - Updated CONFIG with 15 pattern categories
2026-02-05 19:49:15,556 - INFO - High confidence patterns: ['sex_stratification', 'sex_subgroup', 'sex_interaction', 'excludes_pregnant_women']


✓ Patterns loaded into CONFIG


In [5]:
# ============================================================================
# PDF Text Extraction Functions
# ============================================================================

def extract_text_from_pdf_pymupdf(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract text from PDF using PyMuPDF with page tracking.
    
    Returns:
        List of dicts with keys: page_num, text, bbox_info
    """
    pages_data = []
    
    try:
        doc = fitz.open(pdf_path)
        for page_num in range(len(doc)):
            page = doc[page_num]
            text = page.get_text()
            
            pages_data.append({
                'page_num': page_num + 1,  # 1-indexed for humans
                'text': text,
                'char_count': len(text)
            })
        
        doc.close()
        return pages_data
        
    except Exception as e:
        logger.error(f"Error extracting text from {pdf_path} with PyMuPDF: {e}")
        return []


def extract_text_from_pdf_pdfplumber(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract text from PDF using pdfplumber with page tracking.
    
    Returns:
        List of dicts with keys: page_num, text
    """
    pages_data = []
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text() or ""
                
                pages_data.append({
                    'page_num': page_num + 1,  # 1-indexed
                    'text': text,
                    'char_count': len(text)
                })
        
        return pages_data
        
    except Exception as e:
        logger.error(f"Error extracting text from {pdf_path} with pdfplumber: {e}")
        return []


def extract_text_from_pdf(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract text from PDF using available library.
    """
    if not os.path.exists(pdf_path):
        logger.error(f"PDF file not found: {pdf_path}")
        return []
    
    if PDF_LIBRARY == 'pymupdf':
        return extract_text_from_pdf_pymupdf(pdf_path)
    elif PDF_LIBRARY == 'pdfplumber':
        return extract_text_from_pdf_pdfplumber(pdf_path)
    else:
        logger.error("No PDF library available")
        return []

# ============================================================================
# Section Detection (Optional)
# ============================================================================

SECTION_PATTERNS = {
    'abstract': re.compile(r'\babstract\b', re.IGNORECASE),
    'introduction': re.compile(r'\bintroduction\b', re.IGNORECASE),
    'background': re.compile(r'\bbackground\b', re.IGNORECASE),
    'methods': re.compile(r'\b(methods?|methodology)\b', re.IGNORECASE),
    'results': re.compile(r'\bresults?\b', re.IGNORECASE),
    'discussion': re.compile(r'\bdiscussion\b', re.IGNORECASE),
    'conclusions': re.compile(r'\bconclusions?\b', re.IGNORECASE),
    'recommendations': re.compile(r'\brecommendations?\b', re.IGNORECASE),
}


def detect_section(
    text_before_match: str,
    config: Dict[str, Any],
    window_size: int = 200
) -> Optional[str]:
    """
    Try to detect which section a match occurred in by looking at text before match.
    
    Args:
        text_before_match: Text preceding the match
        config: Configuration dictionary (contains section_patterns)
        window_size: How many characters to look back
        
    Returns:
        Section name or None if not detected
    """
    # Get section patterns from config
    section_patterns = config.get('section_patterns', {})
    
    if not section_patterns:
        return None
    
    lookback_text = text_before_match[-window_size:].lower()
    
    # Look for section headers in reverse priority (more specific first)
    for section_name, pattern in section_patterns.items():
        if pattern.search(lookback_text):
            return section_name
    
    return None



# ============================================================================
# Pattern Matching and Snippet Extraction
# ============================================================================

def search_patterns_in_text(
    text: str,
    patterns: List[re.Pattern],
    pattern_category: str,
    page_num: int,
    guideline_id: str,
    config: Dict[str, Any],
    col_guideline_id: str = 'PMID',
    max_snippets: Optional[int] = None,
    snippet_context_chars: Optional[int] = None
) -> Tuple[bool, List[Dict[str, Any]]]:
    """
    Search for patterns in text and extract snippets with context.
    
    Args:
        text: Text to search
        patterns: List of compiled regex patterns
        pattern_category: Category name for these patterns
        page_num: Page number for this text
        guideline_id: Guideline identifier
        config: Configuration dictionary
        col_guideline_id: Name of guideline ID column (default: 'PMID')
        max_snippets: Maximum snippets to capture (uses config if None)
        snippet_context_chars: Context characters around match (uses config if None)
    
    Returns:
        (found_any, list_of_snippet_dicts)
    """
    # global CONFIG, COL_GUIDELINE_ID
    
    # Get parameters from config if not provided
    if max_snippets is None:
        max_snippets = config.get('max_snippets_per_pattern', 5)
    if snippet_context_chars is None:
        snippet_context_chars = config.get('snippet_context_chars', 150)
    
    found = False
    snippets = []
    snippet_count = 0
    
    for pattern in patterns:
        if snippet_count >= max_snippets:
            break
            
        for match in pattern.finditer(text):
            if snippet_count >= max_snippets:
                break
                
            found = True
            
            # Extract snippet with context
            start = max(0, match.start() - snippet_context_chars)
            end = min(len(text), match.end() + snippet_context_chars)
            snippet_text = text[start:end].strip()
            
            # Clean up snippet (remove excessive whitespace)
            snippet_text = re.sub(r'\s+', ' ', snippet_text)
            
            # Try to detect section
            section = None
            if config.get('section_detection', True):
                text_before = text[:match.start()]
                section = detect_section(text_before, config)
            
            snippets.append({
                col_guideline_id: guideline_id,
                'pattern_category': pattern_category,
                'page_num': page_num,
                'section': section,
                'matched_text': match.group(),
                'snippet_text': snippet_text,
                'char_position': match.start(),
                                # ============ ADD THESE REVIEW COLUMNS ============
                'is_false_positive': '',  # Researchers fill: TRUE/FALSE/leave blank
                'reviewer_notes': '',     # Optional notes about why it's false positive
                'reviewed_by': '',        # Optional: reviewer initials
                'review_date': ''         # Optional: when reviewed
                        })
            
            snippet_count += 1
    
    return found, snippets


def analyze_guideline_fulltext(
    guideline_id: str,
    pdf_path: str,
    config: Dict[str, Any],
    extract_text_function,  # Pass the PDF extraction function
    pattern_categories: Optional[Dict[str, List[re.Pattern]]] = None,
    high_confidence_patterns: Optional[List[str]] = None,
    col_guideline_id: str = 'PMID',
    col_pdf_filename: str = 'PDF File Name',
    max_snippets_per_pattern: Optional[int] = None
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    """
    Analyze full text of a guideline PDF for sex-based characteristics.
    
    Args:
        guideline_id: Unique identifier for the guideline
        pdf_path: Path to PDF file
        config: Configuration dictionary
        extract_text_function: Function to extract text from PDF
        pattern_categories: Dict of pattern categories (uses config if None)
        high_confidence_patterns: List of high-confidence pattern names (uses config if None)
        col_guideline_id: Name of guideline ID column
        col_pdf_filename: Name of PDF filename column
        max_snippets_per_pattern: Max snippets per pattern (uses config if None)
    
    Returns:
        (analysis_dict, snippets_list)
    """
     # global COL_GUIDELINE_ID, COL_PDF_FILENAME, PATTERN_CATEGORIES, HIGH_CONFIDENCE_PATTERNS

    # Get parameters from config if not provided
    if pattern_categories is None:
        pattern_categories = config.get('pattern_categories', {})
    if high_confidence_patterns is None:
        high_confidence_patterns = config.get('high_confidence_patterns', [])
    if max_snippets_per_pattern is None:
        max_snippets_per_pattern = config.get('max_snippets_per_pattern', 20)
    
    # Initialize analysis results
    analysis = {
        col_guideline_id: guideline_id,
        col_pdf_filename: os.path.basename(pdf_path),
        'pages_processed': 0,
        'total_chars': 0,
        'processing_status': 'pending',
        'error_message': None,
    }
    
    # Initialize all pattern categories as False
    for category in pattern_categories.keys():
        analysis[f'fulltext_{category}'] = False
    
    all_snippets = []
    
    # Extract text from PDF
    logger.info(f"Processing: {guideline_id} - {os.path.basename(pdf_path)}")
    pages_data = extract_text_function(pdf_path)
    
    if not pages_data:
        analysis['processing_status'] = 'error'
        analysis['error_message'] = 'Failed to extract text from PDF'
        logger.warning(f"Failed to extract text: {guideline_id}")
        return analysis, all_snippets
    
    analysis['pages_processed'] = len(pages_data)
    analysis['total_chars'] = sum(p['char_count'] for p in pages_data)
    
    # Search each page for patterns
    for page_data in pages_data:
        page_num = page_data['page_num']
        text = page_data['text']
        
        if not text or len(text) < 10:  # Skip empty/near-empty pages
            continue
        
        # Search for each pattern category
        for category, patterns in pattern_categories.items():
            found, snippets = search_patterns_in_text(
                text=text,
                patterns=patterns,
                pattern_category=category,
                page_num=page_num,
                guideline_id=guideline_id,
                config=config,
                col_guideline_id=col_guideline_id,
                max_snippets=max_snippets_per_pattern,
                snippet_context_chars=config.get('snippet_context_chars', 200)
            )
            
            if found:
                analysis[f'fulltext_{category}'] = True
                all_snippets.extend(snippets)
    
    # Calculate high-confidence metrics
    high_conf_snippets = [s for s in all_snippets 
                          if s['pattern_category'] in high_confidence_patterns]
    analysis['fulltext_high_confidence_snippet_count'] = len(high_conf_snippets)
    
    # Check if any snippets are in recommendation sections
    rec_sections = ['recommendations', 'conclusions', 'discussion']
    has_rec_sections = any(s.get('section') in rec_sections for s in all_snippets)
    analysis['fulltext_in_recommendations_section'] = has_rec_sections
    
    analysis['processing_status'] = 'success'
    analysis['total_snippets_captured'] = len(all_snippets)
    
    return analysis, all_snippets


# ============================================================================
# Main Processing Pipeline
# ============================================================================

# def load_guideline_metadata() -> pd.DataFrame:
#     """
#     Load guideline metadata from CSV.
    
#     Expected columns are defined in the configuration section:
#         - COL_GUIDELINE_ID: PubMed ID of the guideline
#         - COL_PDF_FILENAME: Name of the PDF file (or full path)
#     """
#     if not os.path.exists(INPUT_CSV_PATH):  # ← UPDATED
#         logger.error(f"Guideline CSV not found: {INPUT_CSV_PATH}")
#         raise FileNotFoundError(f"Missing: {INPUT_CSV_PATH}")
    
#     df = pd.read_csv(INPUT_CSV_PATH, dtype=str)  # ← UPDATED
    
#     # Check for required columns
#     required_cols = [COL_GUIDELINE_ID]  # ← UPDATED - uses config
#     missing_cols = [col for col in required_cols if col not in df.columns]
    
#     if missing_cols:
#         logger.error(f"Missing required columns: {missing_cols}")
#         raise ValueError(f"CSV missing columns: {missing_cols}")
    
#     # Try to find PDF filename column (could be named differently)
#     pdf_col = None
#     for col in PDF_FILENAME_ALTERNATIVES:  # ← UPDATED - uses config
#         if col in df.columns:
#             pdf_col = col
#             break
    
#     if pdf_col:
#         logger.info(f"Using '{pdf_col}' column for PDF filenames")
#         if pdf_col != COL_PDF_FILENAME:  # ← UPDATED
#             df[COL_PDF_FILENAME] = df[pdf_col]  # ← UPDATED
#     else:
#         logger.warning("No PDF filename column found - will construct from ID column")
#         # Assume PDFs are named by ID
#         df[COL_PDF_FILENAME] = df[COL_GUIDELINE_ID] + '.pdf'  # ← UPDATED
    
#     logger.info(f"Loaded {len(df)} guidelines from {INPUT_CSV_PATH}")
#     return df

def load_guideline_metadata() -> pd.DataFrame:
    """
    Load guideline metadata from CSV.

    Required columns (from config):
      - COL_GUIDELINE_ID: e.g., 'PMID'
      - COL_PDF_FILENAME: e.g., 'PDF File Name'
    """
    #global INPUT_CSV_PATH, COL_GUIDELINE_ID, COL_PDF_FILENAME

    if not os.path.exists(INPUT_CSV_PATH):
        logger.error(f"Guideline CSV not found: {INPUT_CSV_PATH}")
        raise FileNotFoundError(f"Missing: {INPUT_CSV_PATH}")

    # Read as strings so PMIDs don't become floats and filenames stay intact
    df = pd.read_csv(INPUT_CSV_PATH, dtype=str)

    # Require BOTH the ID and the PDF filename columns
    required_cols = [COL_GUIDELINE_ID, COL_PDF_FILENAME]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        logger.error(f"Missing required columns: {missing_cols}. Found: {list(df.columns)}")
        raise ValueError(f"CSV missing columns: {missing_cols}")

    # Clean up whitespace
    df[COL_GUIDELINE_ID] = df[COL_GUIDELINE_ID].astype(str).str.strip()
    df[COL_PDF_FILENAME] = df[COL_PDF_FILENAME].astype(str).str.strip()

    # Optional: warn if any filenames are blank
    blank_pdf = df[COL_PDF_FILENAME].eq("") | df[COL_PDF_FILENAME].str.lower().eq("nan")
    if blank_pdf.any():
        logger.warning(f"{blank_pdf.sum()} rows have blank PDF filenames in '{COL_PDF_FILENAME}'")

    logger.info(f"Loaded {len(df)} guidelines from {INPUT_CSV_PATH}")

    # After loading the CSV, add:
    # Ensure all filenames have .pdf extension
    mask = ~df[COL_PDF_FILENAME].str.lower().str.endswith('.pdf', na=False)
    if mask.any():
        logger.warning(f"Found {mask.sum()} filenames without .pdf extension - adding it")
        df.loc[mask, COL_PDF_FILENAME] = df.loc[mask, COL_PDF_FILENAME] + '.pdf'
    
    return df


def process_all_guidelines(
    guidelines_df: pd.DataFrame,
    config: Dict[str, Any],
    limit: Optional[int] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Process all guidelines and return results.
    
    Args:
        guidelines_df: DataFrame with guideline metadata
        config: Configuration dictionary
        limit: Optional limit for testing (process only first N guidelines)
        
    Returns:
        (analysis_df, snippets_df)
    """
    # Extract from config
    col_guideline_id = config['col_guideline_id']
    col_pdf_filename = config['col_pdf_filename']
    pdf_directory = config['pdf_directory']
    pattern_categories = config['pattern_categories']
    high_confidence_patterns = config['high_confidence_patterns']
    
    all_analyses = []
    all_snippets = []
    
    # Limit for testing
    if limit:
        logger.info(f"Processing limited to first {limit} guidelines for testing")
        guidelines_df = guidelines_df.head(limit)
    
    total = len(guidelines_df)
    
    for idx, row in guidelines_df.iterrows():
        guideline_id = str(row[col_guideline_id])       # ← Use local variable
        pdf_filename = str(row[col_pdf_filename])       # ← Use local variable
        
        # Construct full path
        pdf_path = os.path.join(pdf_directory, pdf_filename)  # ← Use local variable
        
        # Check if PDF exists
        if not os.path.exists(pdf_path):
            logger.warning(f"PDF not found ({idx+1}/{total}): {pdf_path}")
            # Still record the analysis with error status
            analysis = {
                col_guideline_id: guideline_id,         # ← Use local variable
                col_pdf_filename: pdf_filename,         # ← Use local variable
                'processing_status': 'error',
                'error_message': 'PDF file not found',
                'pages_processed': 0,
                'total_chars': 0,
            }
            for category in pattern_categories.keys():  # ← Use local variable
                analysis[f'fulltext_{category}'] = np.nan
            all_analyses.append(analysis)
            continue
        
        # Process guideline
        logger.info(f"Processing {idx+1}/{total}: {guideline_id}")
        analysis, snippets = analyze_guideline_fulltext(
            guideline_id=guideline_id,
            pdf_path=pdf_path,
            config=config,                              # ← Only once, use parameter
            extract_text_function=extract_text_from_pdf,
            pattern_categories=pattern_categories,      # ← Use local variable
            high_confidence_patterns=high_confidence_patterns,  # ← Use local variable
            col_guideline_id=col_guideline_id,          # ← Use local variable
            col_pdf_filename=col_pdf_filename,          # ← Use local variable
            max_snippets_per_pattern=3
        )
        
        all_analyses.append(analysis)
        all_snippets.extend(snippets)
    
    # Convert to DataFrames
    analysis_df = pd.DataFrame(all_analyses)
    snippets_df = pd.DataFrame(all_snippets) if all_snippets else pd.DataFrame()
    
    logger.info(f"\nProcessing complete:")
    logger.info(f"  Total guidelines: {len(analysis_df)}")
    logger.info(f"  Successfully processed: {(analysis_df['processing_status'] == 'success').sum()}")
    logger.info(f"  Errors: {(analysis_df['processing_status'] == 'error').sum()}")
    logger.info(f"  Total snippets captured: {len(snippets_df)}")
    
    return analysis_df, snippets_df

def save_results(analysis_df: pd.DataFrame, snippets_df: pd.DataFrame):
    """Save analysis results to CSV files."""
    
    # Main analysis file
    analysis_file = os.path.join(OUTPUT_FOLDER, 'guideline_fulltext_sex_analysis.csv')
    analysis_df.to_csv(analysis_file, index=False)
    logger.info(f"Saved analysis results: {analysis_file}")
    
    # Snippets file (if any snippets were captured)
    if len(snippets_df) > 0:
        snippets_file = os.path.join(OUTPUT_FOLDER, 'guideline_fulltext_snippets.csv')
        snippets_df.to_csv(snippets_file, index=False)
        logger.info(f"Saved snippets: {snippets_file}")
    
    # Summary statistics
    summary_file = os.path.join(OUTPUT_FOLDER, 'guideline_fulltext_summary.txt')
    with open(summary_file, 'w') as f:
        f.write("Guideline Full-Text Analysis Summary\n")
        f.write("=" * 70 + "\n\n")
        f.write(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total guidelines: {len(analysis_df)}\n")
        f.write(f"Successfully processed: {(analysis_df['processing_status'] == 'success').sum()}\n")
        f.write(f"Errors: {(analysis_df['processing_status'] == 'error').sum()}\n\n")
        
        # Pattern category statistics
        f.write("Pattern Category Detection Rates:\n")
        f.write("-" * 70 + "\n")
        for category in PATTERN_CATEGORIES.keys():
            col_name = f'fulltext_{category}'
            if col_name in analysis_df.columns:
                count = analysis_df[col_name].sum()
                pct = (count / len(analysis_df)) * 100
                f.write(f"{category:40s}: {count:4d} ({pct:5.1f}%)\n")
        
        f.write("\n")
        f.write(f"Total snippets captured: {len(snippets_df)}\n")
        
        if len(snippets_df) > 0:
            f.write("\nSnippets by category:\n")
            f.write("-" * 70 + "\n")
            snippet_counts = snippets_df['pattern_category'].value_counts()
            for category, count in snippet_counts.items():
                f.write(f"{category:40s}: {count:4d}\n")
    
    logger.info(f"Saved summary: {summary_file}")

In [6]:
# ============================================================================
# Main Execution
# ============================================================================


print(f"\n{'='*70}")
print("STARTING GUIDELINE FULL-TEXT ANALYSIS")
print(f"{'='*70}\n")

# Check if PDF library is available
if PDF_LIBRARY is None:
    print("❌ ERROR: No PDF library available!")
    print("Please install PyMuPDF: pip install PyMuPDF")
    exit(1)

# Load guideline metadata
print("Loading guideline metadata...")
guidelines_df = load_guideline_metadata()
print(f"✓ Loaded {len(guidelines_df)} guidelines\n")

# Check if PDF folder exists
if not os.path.exists(PDF_FOLDER):
    print(f"⚠️ WARNING: PDF folder not found: {PDF_FOLDER}")
    print(f"Creating folder: {PDF_FOLDER}")
    os.makedirs(PDF_FOLDER, exist_ok=True)

# Count available PDFs
available_pdfs = 0
for _, row in guidelines_df.iterrows():
    pdf_path = os.path.join(PDF_DIRECTORY, str(row[COL_PDF_FILENAME]))
    if os.path.exists(pdf_path):
        available_pdfs += 1

print(f"PDF directory: {PDF_DIRECTORY}")
print(f"Available PDFs: {available_pdfs} / {len(guidelines_df)}\n")

if available_pdfs == 0:
    print("⚠️ WARNING: No PDF files found in the PDF directory!")
    print("Please add PDF files to proceed with analysis.")
    exit(0)

# Ask for confirmation before processing
print(f"Ready to process {available_pdfs} guidelines.")
response = input("Proceed with full analysis? (y/n) or enter number for test run: ")

limit = None
if response.lower() == 'y':
    limit = None
elif response.lower() == 'n':
    print("Analysis cancelled.")
    exit(0)
else:
    try:
        limit = int(response)
        print(f"Processing first {limit} guidelines as test run...")
    except ValueError:
        print("Invalid input. Analysis cancelled.")
        exit(0)

# Process all guidelines
print(f"\n{'='*70}")
print("PROCESSING GUIDELINES")
print(f"{'='*70}\n")

analysis_df, snippets_df = process_all_guidelines(
    guidelines_df, 
    config=CONFIG, 
    limit=limit
)

# ============================================================================
# ADD SCENARIO MEMBERSHIP FLAGS FROM REFERENCE/NCT ANALYSIS
# ============================================================================

if len(analysis_df) > 0:
    print(f"\n{'='*70}")
    print("Adding scenario membership flags from reference/NCT analysis...")
    print(f"{'='*70}\n")
    
    # Path to your Phase 8 enhanced file
    phase8_enhanced_file = os.path.join(
        'output', 
        'all_final_guidelines',
        'phase8_FULLY_EXPLODED_UNIVERSE_all_guidelines_all_references_all_nct_with_scenario_membership.csv'
    )
    
    if os.path.exists(phase8_enhanced_file):
        print(f"Loading scenario data from: {phase8_enhanced_file}")
        
        # Load just the columns we need
        scenario_cols = [
            'guideline_pmid',
            'in_S1_All_References',
            'in_S2_PubMed_Trials_All_NCTs',
            'in_S3_PubMed_Trials_Primary_NCT_Only'
        ]
        
        df_scenarios = pd.read_csv(phase8_enhanced_file, usecols=scenario_cols, low_memory=False)
        
        print(f"  Loaded {len(df_scenarios):,} rows")
        
        # Convert scenario columns to boolean
        for col in ['in_S1_All_References', 'in_S2_PubMed_Trials_All_NCTs', 'in_S3_PubMed_Trials_Primary_NCT_Only']:
            if col in df_scenarios.columns:
                df_scenarios[col] = df_scenarios[col].map({
                    'True': True, True: True, 'TRUE': True,
                    'False': False, False: False, 'FALSE': False
                })
        
        # Aggregate by guideline: if guideline had ANY row in a scenario, mark as TRUE
        print("  Aggregating scenarios by guideline...")
        scenario_by_guideline = df_scenarios.groupby('guideline_pmid').agg({
            'in_S1_All_References': 'any',
            'in_S2_PubMed_Trials_All_NCTs': 'any',
            'in_S3_PubMed_Trials_Primary_NCT_Only': 'any'
        }).reset_index()
        
        print(f"  Aggregated to {len(scenario_by_guideline):,} unique guidelines")
        
        # Show scenario counts
        for col in ['in_S1_All_References', 'in_S2_PubMed_Trials_All_NCTs', 'in_S3_PubMed_Trials_Primary_NCT_Only']:
            count = scenario_by_guideline[col].sum()
            pct = (count / len(scenario_by_guideline) * 100)
            print(f"    {col}: {count:,} guidelines ({pct:.1f}%)")
        
        # Merge with analysis results
        analysis_df[COL_GUIDELINE_ID] = analysis_df[COL_GUIDELINE_ID].astype(str)
        scenario_by_guideline['guideline_pmid'] = scenario_by_guideline['guideline_pmid'].astype(str)
        
        print("\n  Merging with full-text analysis results...")
        analysis_df = analysis_df.merge(
            scenario_by_guideline,
            left_on=COL_GUIDELINE_ID,
            right_on='guideline_pmid',
            how='left'
        )
        
        # Drop duplicate column if created
        if 'guideline_pmid' in analysis_df.columns and 'guideline_pmid' != COL_GUIDELINE_ID:
            analysis_df = analysis_df.drop(columns=['guideline_pmid'])
        
        # Fill NaN with False
        for col in ['in_S1_All_References', 'in_S2_PubMed_Trials_All_NCTs', 'in_S3_PubMed_Trials_Primary_NCT_Only']:
            if col in analysis_df.columns:
                analysis_df[col] = analysis_df[col].fillna(False)
        
        print(f"  ✓ Scenario flags added to {len(analysis_df)} guidelines")
        
    else:
        print(f"⚠️ Phase 8 file not found: {phase8_enhanced_file}")
        print("  Creating placeholder columns (all FALSE)")
        analysis_df['in_S1_All_References'] = False
        analysis_df['in_S2_PubMed_Trials_All_NCTs'] = False
        analysis_df['in_S3_PubMed_Trials_Primary_NCT_Only'] = False
    
    print(f"{'='*70}\n")

# ============================================================================
# ADD GUIDELINE GROUPINGS
# ============================================================================

if len(analysis_df) > 0:
    print(f"\n{'='*70}")
    print("Adding guideline groupings...")
    print(f"{'='*70}\n")
    
    # Define the guideline groupings
    GROUPING_DEFINITIONS = {
        'grouping_all_final_guidelines': {
            'pmids': 'ALL',
            'description': 'All guidelines in the dataset'
        },
        
        'grouping_only_guidelines': {
            'pmids': [
                41582814, 41411375, 40997152, 40811497, 40997143, 40997146, 40014670, 
                39429201, 39530204, 39540278, 38033089, 37970724, 38108133, 38743805, 
                39316661, 38718139, 37471501, 37721023, 37212182, 35579034, 35363499, 
                35363500, 34882435, 34882436, 36322642, 34024115, 34709879, 34709928, 
                34024117, 33332150, 33332149, 33215931, 33215938, 30879355, 30879339, 
                31722559, 31722551, 31722546, 31724451, 30565953, 30586772, 30586771, 
                30892927, 30586767, 30586768, 30586774, 30686041, 31722552, 31722563, 
                31662037, 30586773, 30586769, 30586770, 30586775, 29084731, 29084733, 
                29114009, 30571264, 29114008, 29084732, 29133356, 29133354, 30571262, 
                29367334, 29367333, 29367332, 29133355, 28455343, 28280231, 28280232, 
                28280230, 26399661, 27208049, 27026019, 26399663, 26399662, 26490017, 
                27026020, 27208050, 26399660, 27145936, 27789558, 26637530, 26534956, 
                26472998, 25249585, 25249586, 24677315, 25212466, 24222015, 25070666, 
                25085961, 25085962, 24222018, 25092464, 24682348, 24222016, 24222017, 
                24589853, 24589852, 24682347, 25085964
            ],
            'description': 'Guidelines only (clinical practice guidelines)'
        },
        
        'grouping_only_aha_guidelines': {
            'pmids': [
                41582814, 41411375, 40811497, 40014670, 39429201, 39540278, 38033089, 
                38743805, 39316661, 38718139, 37471501, 37212182, 35579034, 35363499, 
                34882435, 36322642, 34709879, 34024117, 33332150, 33215931, 30879355, 
                30586772, 30586767, 30586774, 29084731, 29133356, 29367334, 28280231, 
                26399663, 27145936, 26534956, 25249585, 24222015, 25085961, 24222018, 
                24222016, 24222017, 24589853, 24682347
            ],
            'description': 'AHA guidelines only (American Heart Association)'
        },
        
        'grouping_only_other_statements': {
            'pmids': [
                35737748, 39045706, 26472853, 34196223, 34196222, 27143685, 26666514, 
                36648070, 26696642, 27528691, 27162236
            ],
            'description': 'Other statements (non-guideline, non-scientific)'
        },
        
        'grouping_only_scientific_statements': {
            'pmids': [
                # ... (your full list) ...
            ],
            'description': 'Scientific statements (position papers, consensus statements)'
        }
    }
    
    # Add grouping columns
    for grouping_name, grouping_config in GROUPING_DEFINITIONS.items():
        
        if grouping_config['pmids'] == 'ALL':
            analysis_df[grouping_name] = True
            count = len(analysis_df)
        else:
            pmid_list = [int(p) for p in grouping_config['pmids']]
            analysis_df[grouping_name] = analysis_df[COL_GUIDELINE_ID].astype(int).isin(pmid_list)
            count = analysis_df[grouping_name].sum()
        
        pct = (count / len(analysis_df) * 100)
        
        print(f"  ✓ {grouping_name}")
        print(f"      {grouping_config['description']}")
        print(f"      Guidelines: {count:,} ({pct:.1f}%)")
        print()
    
    print(f"{'='*70}\n")

# ============================================================================
# SAVE RESULTS
# ============================================================================

if len(analysis_df) > 0:
    print(f"\n{'='*70}")
    print("SAVING RESULTS")
    print(f"{'='*70}\n")
    
    save_results(analysis_df, snippets_df)
    
    print(f"\n{'='*70}")
    print("ANALYSIS COMPLETE")
    print(f"{'='*70}\n")
    
    # Display summary
    print("\nSummary:")
    print(f"  Total guidelines analyzed: {len(analysis_df)}")
    print(f"  Successful: {(analysis_df['processing_status'] == 'success').sum()}")
    print(f"  Errors: {(analysis_df['processing_status'] == 'error').sum()}")
    print(f"  Total snippets: {len(snippets_df)}")
    
    # Show pattern detection rates
    if len(analysis_df[analysis_df['processing_status'] == 'success']) > 0:
        print("\nPattern detection rates:")
        for category in ['sex_differences', 'sex_stratification', 'sex_subgroup', 
                        'pregnancy_related', 'menopause_related']:
            col_name = f'fulltext_{category}'
            if col_name in analysis_df.columns:
                count = analysis_df[col_name].sum()
                pct = (count / len(analysis_df)) * 100
                print(f"  {category}: {count} guidelines ({pct:.1f}%)")
    
    print("\nOutput files:")
    print(f"  - {os.path.join(OUTPUT_FOLDER, 'guideline_fulltext_sex_analysis.csv')}")
    print(f"  - {os.path.join(OUTPUT_FOLDER, 'guideline_fulltext_snippets.csv')}")
    print(f"  - {os.path.join(OUTPUT_FOLDER, 'guideline_fulltext_summary.txt')}")

2026-02-05 19:49:55,305 - INFO - Loaded 496 guidelines from data\all_final_guidelines.csv
2026-02-05 19:49:55,307 - WARNING - Found 302 filenames without .pdf extension - adding it



STARTING GUIDELINE FULL-TEXT ANALYSIS

Loading guideline metadata...
✓ Loaded 496 guidelines

PDF directory: data\guidelines full text
Available PDFs: 496 / 496

Ready to process 496 guidelines.


Proceed with full analysis? (y/n) or enter number for test run:  y


2026-02-05 19:49:58,698 - INFO - Processing 1/496: 37650292
2026-02-05 19:49:58,699 - INFO - Processing: 37650292 - Abdalla-2023-Implementation Strategies to Impr.pdf



PROCESSING GUIDELINES



2026-02-05 19:49:59,122 - INFO - Processing 2/496: 26534956
2026-02-05 19:49:59,123 - INFO - Processing: 26534956 - Abman-2015-Pediatric Pulmonary Hypertension_ G.pdf
2026-02-05 19:50:00,669 - INFO - Processing 3/496: 26527716
2026-02-05 19:50:00,671 - INFO - Processing: 26527716 - Ackerman-2015-Eligibility and Disqualification.pdf
2026-02-05 19:50:00,781 - INFO - Processing 4/496: 37732422
2026-02-05 19:50:00,782 - INFO - Processing: 37732422 - Addison-2023-Cardiovascular Imaging in Contemp.pdf
2026-02-05 19:50:01,092 - INFO - Processing 5/496: 37377045
2026-02-05 19:50:01,092 - INFO - Processing: 37377045 - Addison-2023-Equity in Cardio-Oncology Care an.pdf
2026-02-05 19:50:01,335 - INFO - Processing 6/496: 37698007
2026-02-05 19:50:01,336 - INFO - Processing: 37698007 - Agarwala-2023-Implementation of Prevention Sci.pdf
2026-02-05 19:50:01,616 - INFO - Processing 7/496: 38450477
2026-02-05 19:50:01,618 - INFO - Processing: 38450477 - Aggarwal-2024-Status and Future Directions for.pd


Adding scenario membership flags from reference/NCT analysis...

⚠️ Phase 8 file not found: output\all_final_guidelines\phase8_FULLY_EXPLODED_UNIVERSE_all_guidelines_all_references_all_nct_with_scenario_membership.csv
  Creating placeholder columns (all FALSE)


Adding guideline groupings...

  ✓ grouping_all_final_guidelines
      All guidelines in the dataset
      Guidelines: 496 (100.0%)

  ✓ grouping_only_guidelines
      Guidelines only (clinical practice guidelines)
      Guidelines: 102 (20.6%)

  ✓ grouping_only_aha_guidelines
      AHA guidelines only (American Heart Association)
      Guidelines: 39 (7.9%)

  ✓ grouping_only_other_statements
      Other statements (non-guideline, non-scientific)
      Guidelines: 11 (2.2%)

  ✓ grouping_only_scientific_statements
      Scientific statements (position papers, consensus statements)
      Guidelines: 0 (0.0%)



SAVING RESULTS



2026-02-05 19:54:03,007 - INFO - Saved snippets: output\all_final_guidelines\full_text_analysis\guideline_fulltext_snippets.csv
2026-02-05 19:54:03,018 - INFO - Saved summary: output\all_final_guidelines\full_text_analysis\guideline_fulltext_summary.txt



ANALYSIS COMPLETE


Summary:
  Total guidelines analyzed: 496
  Successful: 496
  Errors: 0
  Total snippets: 15250

Pattern detection rates:
  sex_differences: 446 guidelines (89.9%)
  sex_stratification: 10 guidelines (2.0%)
  sex_subgroup: 16 guidelines (3.2%)
  pregnancy_related: 210 guidelines (42.3%)
  menopause_related: 73 guidelines (14.7%)

Output files:
  - output\all_final_guidelines\full_text_analysis\guideline_fulltext_sex_analysis.csv
  - output\all_final_guidelines\full_text_analysis\guideline_fulltext_snippets.csv
  - output\all_final_guidelines\full_text_analysis\guideline_fulltext_summary.txt


In [4]:
# ============================================================================
# RECALCULATE ANALYSIS BOOLEANS FROM VERIFIED SNIPPETS
# ============================================================================

import pandas as pd
import os
import numpy as np

def recalculate_analysis_from_verified_snippets(
    snippets_file: str,
    analysis_file: str,
    output_file: str = None
) -> pd.DataFrame:
    """
    Recalculate guideline analysis booleans based on verified snippets.
    
    Logic:
    - If ALL snippets for a category are marked FALSE → set boolean to FALSE
    - If ANY snippet for a category is NOT false positive → keep boolean TRUE
    - If category has no snippets → keep original boolean (might be from different source)
    
    Args:
        snippets_file: Path to reviewed snippets CSV
        analysis_file: Path to original analysis CSV
        output_file: Path to save updated analysis (if None, creates *_VERIFIED.csv)
    
    Returns:
        Updated analysis DataFrame
    """
    
    print(f"\n{'='*70}")
    print("RECALCULATING ANALYSIS FROM VERIFIED SNIPPETS")
    print(f"{'='*70}\n")
    
    # Load data
    print(f"Loading snippets: {snippets_file}")
    snippets_df = pd.read_csv(snippets_file, dtype={'is_false_positive': str})
    
    print(f"Loading analysis: {analysis_file}")
    analysis_df = pd.read_csv(analysis_file)
    
    print(f"  Snippets: {len(snippets_df):,} rows")
    print(f"  Guidelines: {len(analysis_df):,} rows")
    
    # Normalize the is_false_positive column
    print("\nNormalizing review flags...")
    snippets_df['is_false_positive'] = snippets_df['is_false_positive'].astype(str).str.strip().str.upper()
    snippets_df['is_false_positive_bool'] = snippets_df['is_false_positive'].map({
        'TRUE': True, 'T': True, 'YES': True, 'Y': True, '1': True,
        'FALSE': False, 'F': False, 'NO': False, 'N': False, '0': False,
        'NAN': False, '': False  # Blank = not false positive
    })
    
    # Count reviewed snippets
    reviewed_count = (snippets_df['is_false_positive'].isin(['TRUE', 'FALSE', 'T', 'F', 'YES', 'NO', 'Y', 'N', '1', '0'])).sum()
    print(f"  Snippets with review flags: {reviewed_count:,} / {len(snippets_df):,}")
    
    if reviewed_count == 0:
        print("\n⚠️ WARNING: No snippets have been reviewed yet!")
        print("  The 'is_false_positive' column is empty.")
        print("  Returning original analysis unchanged.")
        return analysis_df
    
    # Get all pattern categories
    pattern_categories = snippets_df['pattern_category'].unique()
    print(f"\nPattern categories to recalculate: {len(pattern_categories)}")
    
    # Track changes
    changes = []
    
    # For each guideline, recalculate each pattern category
    print("\nRecalculating booleans...")
    for pmid in analysis_df[COL_GUIDELINE_ID].unique():
        pmid_snippets = snippets_df[snippets_df[COL_GUIDELINE_ID] == pmid]
        
        for category in pattern_categories:
            category_snippets = pmid_snippets[pmid_snippets['pattern_category'] == category]
            
            if len(category_snippets) == 0:
                continue  # No snippets for this category, keep original value
            
            # Check if ALL snippets are marked as false positives
            all_false_positive = category_snippets['is_false_positive_bool'].all()
            
            # Get current boolean value
            col_name = f'fulltext_{category}'
            if col_name not in analysis_df.columns:
                continue
            
            current_value = analysis_df.loc[analysis_df[COL_GUIDELINE_ID] == pmid, col_name].iloc[0]
            
            # Determine new value
            if all_false_positive:
                new_value = False
            else:
                new_value = True  # At least one snippet is NOT false positive
            
            # Update if changed
            if current_value != new_value:
                analysis_df.loc[analysis_df[COL_GUIDELINE_ID] == pmid, col_name] = new_value
                changes.append({
                    'PMID': pmid,
                    'category': category,
                    'old_value': current_value,
                    'new_value': new_value,
                    'total_snippets': len(category_snippets),
                    'false_positives': category_snippets['is_false_positive_bool'].sum()
                })
    
    print(f"  ✓ Recalculation complete")
    
    # Show summary of changes
    if len(changes) > 0:
        print(f"\n{'='*70}")
        print(f"CHANGES MADE: {len(changes)} booleans updated")
        print(f"{'='*70}\n")
        
        changes_df = pd.DataFrame(changes)
        print(changes_df.to_string(index=False))
        
        # Save changes log
        changes_file = analysis_file.replace('.csv', '_CHANGES_LOG.csv')
        changes_df.to_csv(changes_file, index=False)
        print(f"\n✓ Changes log saved: {changes_file}")
    else:
        print(f"\n✓ No changes needed - all verified snippets confirmed original flags")
    
    # Save updated analysis
    if output_file is None:
        output_file = analysis_file.replace('.csv', '_VERIFIED.csv')
    
    analysis_df.to_csv(output_file, index=False)
    print(f"\n✓ Updated analysis saved: {output_file}")
    
    # Summary statistics
    print(f"\n{'='*70}")
    print("VERIFICATION SUMMARY")
    print(f"{'='*70}\n")
    
    print(f"Snippets reviewed: {reviewed_count:,} / {len(snippets_df):,} ({reviewed_count/len(snippets_df)*100:.1f}%)")
    print(f"False positives identified: {snippets_df['is_false_positive_bool'].sum():,}")
    print(f"Booleans changed: {len(changes)}")
    
    if len(changes) > 0:
        print(f"\nChanges by category:")
        category_changes = pd.DataFrame(changes).groupby('category').size()
        for cat, count in category_changes.items():
            print(f"  {cat}: {count} guidelines")
    
    print(f"\n{'='*70}\n")
    
    return analysis_df


# ============================================================================
# RUN VERIFICATION
# ============================================================================

# Define file paths
SNIPPETS_FILE = os.path.join(OUTPUT_FOLDER, 'guideline_fulltext_snippets.csv')
ANALYSIS_FILE = os.path.join(OUTPUT_FOLDER, 'guideline_fulltext_sex_analysis.csv')
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, 'guideline_fulltext_sex_analysis_VERIFIED.csv')

# Run recalculation
updated_analysis = recalculate_analysis_from_verified_snippets(
    snippets_file=SNIPPETS_FILE,
    analysis_file=ANALYSIS_FILE,
    output_file=OUTPUT_FILE
)

print("✅ Verification complete!")
print("\nNext steps:")
print("  1. Review the changes log")
print("  2. Use the *_VERIFIED.csv file for final analysis")


RECALCULATING ANALYSIS FROM VERIFIED SNIPPETS

Loading snippets: output\all_final_guidelines\full_text_analysis\guideline_fulltext_snippets.csv
Loading analysis: output\all_final_guidelines\full_text_analysis\guideline_fulltext_sex_analysis.csv
  Snippets: 15,250 rows
  Guidelines: 496 rows

Normalizing review flags...


KeyError: 'is_false_positive'